In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import preprocessing

In [ ]:
col_names = ["unit", "time_cycles", "op_set1", "op_set2", "op_set3"] + [f'sensor{i}' for i in range(1, 22)]

df_train= pd.read_csv("CMAPSSData/train_FD001.txt", sep=r"\s+", header=None, names=col_names)
df_test = pd.read_csv("CMAPSSData/test_FD001.txt", sep=r"\s+", header=None, names=col_names)
df_rul = pd.read_csv("CMAPSSData/RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL"])

In [ ]:
train_unit_100 = df_train[df_train["unit"] == 100]
test_unit_100 = df_test[df_test["unit"] == 100]
sensors_to_plot = ['sensor1', 'sensor3', 'sensor20']

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(10, 8), sharex=True)

for i, sensor in enumerate(sensors_to_plot):
    axes[i].plot(train_unit_100["time_cycles"], train_unit_100[sensor], color='blue')
    axes[i].set_ylabel(sensor + " Value")
    axes[i].set_title(sensor)
    axes[i].grid(True) 

axes[-1].set_xlabel("Time Cycles")
plt.tight_layout()
plt.show()

In [ ]:
df_train['RUL'] = df_train.groupby('unit')['time_cycles'].transform(lambda x: x.max() - x) 
#We know max lifetime, so we can use x.max()-x in time cycles for find RUL with grouping by units
df_train

In [ ]:
exclude_cols = [f'sensor_{i}' for i in [1, 5, 6, 10, 16, 18, 19]] #Unnecessary(stable) sensors were removed
feature_cols = df_train.filter(regex='sensor').columns.difference(exclude_cols) #op_set not important for FD001

scaler = preprocessing.StandardScaler() #instead of minmax scaler
df_train[feature_cols] = scaler.fit_transform(df_train[feature_cols])
df_test[feature_cols] = scaler.transform(df_test[feature_cols]) # Old fit was used for prevent data leak 

In [ ]:
windowing_length = 30

def create_train_sequences(df, seq_len):
    sequences, targets = [], []
    for unit in df['unit'].unique():
        engine_data = df[df['unit'] == unit]
        features = engine_data[feature_cols].values
        rul_values = engine_data['RUL'].values

        for i in range(len(features) - seq_len + 1):
            sequences.append(features[i : i + seq_len])
            targets.append(rul_values[i + seq_len - 1]) #RUL values append to targets
            
    return np.array(sequences), np.array(targets)

def get_last_test_windows(df, seq_len):
    last_windows = []
    for unit in df['unit'].unique():
        engine_data = df[df['unit'] == unit]
        features = engine_data[feature_cols].values
        last_windows.append(features[-seq_len:]) #last 30 window important for evaluate RUL
        
    return np.array(last_windows)

In [ ]:
X_train, y_train = create_train_sequences(df_train, windowing_length)
X_test = get_last_test_windows(df_test, windowing_length)

y_test = df_rul.values

"""print(f"X_train: {X_train.shape}") 
print(f"Y_train: {y_train.shape}")   
print(f"X_test: {X_test.shape}")    
print(f"y_test: {y_test.shape}")""" 